In [1]:
# 데이터 불러오기

from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# 학습 데이터 불러오기
train_data = datasets.FashionMNIST(
            root="../Data" , # 어디에 저장할 건지
            train=True, # train 인지 
            download=True,# 다운로드 해야 함
            transform=ToTensor() # 이렇게 바꿔줌
)

In [3]:
# 테스트 데이터 불러오기
test_data = datasets.FashionMNIST(
            root="../Data" , # 어디에 저장할 건지
            train=False,
            download=True,
            transform=ToTensor()
)

In [4]:
# 받은 데이터 확인
print(train_data.data.shape)
print(train_data.targets.shape)
print(test_data.data.shape)
print(test_data.targets.shape)

torch.Size([60000, 28, 28])
torch.Size([60000])
torch.Size([10000, 28, 28])
torch.Size([10000])


In [5]:
# data와 target 분류
train_input = train_data.data
train_target = train_data.targets

test_input = test_data.data
test_target = test_data.targets
# 간단하게 쓰려고 변수 정해준 거 

In [6]:
# 데이터 표준화 및 2차원 행렬
train_scaled = (train_input / 255.0).reshape(-1, 28*28)
test_scaled = (test_input / 255.0).reshape(-1, 28*28)

print(train_scaled.shape)
print(test_scaled.shape)

torch.Size([60000, 784])
torch.Size([10000, 784])


In [7]:
# Train과 Valid
from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(
                                                        train_scaled,
                                                        train_target,
                                                        test_size=0.2,
                                                        random_state=42
)

----
#### 심층 신경망 만들기

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [9]:
# Dataset과 Dataloader 생성

batch_size = 32 # mini batch
train_dataset = TensorDataset(train_scaled, train_target)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # 한번 epoch가 발생했을때 섞어 쓰는 거

val_dataset = TensorDataset(val_scaled, val_target)
val_loader = DataLoader(val_dataset, batch_size=batch_size) # 벨리드나 테스트는 검증이라 셔플을 쓰지 않는다 

---
### 모델 정의

In [10]:
# 입력층 -> 은닉층(활성화함수) -> 출력층


    

class FashionMNISTModel1(nn.Module):
    def __init__(self):
        super(FashionMNISTModel1, self).__init__() # super에 있는 모델을 쓰겠다는 거 
        self.flatten = nn.Flatten() # 층을 펴주는 거 
        self.fc1 = nn.Linear(28*28, 512)
        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(512,10) # 512개 들어와서 10개로 준다 
        self.softmax = nn.Softmax(dim=1) # 앞에는 변수임
    
    def forward(self, x) : 
        x = self.flatten(x) # 들어온 데이터로 층 만든것
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return self.softmax(x)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [12]:
# 모델 , 손실함수, 옵티마이져 초기화

model1 = FashionMNISTModel1().to(device)
criterion1 = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(model1.parameters())

---
#### 모델 훈련

In [13]:
# 학습 함수
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device) # 디바이스 (cpu)로 보냄
        optimizer.zero_grad() # 초기화 시켜주는 거 _옵티마이저가 곱하기 하는거라서 
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    return loss.item()

In [14]:
# 평가함수 
def evaluate(model,val_loader,criterion,device):
    model.eval()
    total_loss = 0 # 전체 손실 합계 
    correct = 0  # 정확하게 예측한 샘플 수 
    total = 0 # 전체 샘플 수
    with torch.no_grad():
        for inputs ,targets in val_loader: # 문제 정답 넣기
            inputs,targets = inputs.to(device),targets.to(device) # 문제 정답 디바이스로 보내기
            outputs = model(inputs)
            loss = criterion(outputs,targets)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss / len(val_loader), correct / total

In [15]:
model1.to(device)

FashionMNISTModel1(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [16]:
# 훈련하기 - 1
num_epochs = 50

for epoch in range(num_epochs):
    train_loss = train(model1, train_loader, criterion1, optimizer1, device)
    print(f'Epoch[{epoch+1:>3} / {num_epochs}], Loss : {train_loss:.4f}')

Epoch[  1 / 50], Loss : 2.2346
Epoch[  2 / 50], Loss : 2.2124
Epoch[  3 / 50], Loss : 2.2014
Epoch[  4 / 50], Loss : 2.1993
Epoch[  5 / 50], Loss : 2.1822
Epoch[  6 / 50], Loss : 2.1911
Epoch[  7 / 50], Loss : 2.1848
Epoch[  8 / 50], Loss : 2.2053
Epoch[  9 / 50], Loss : 2.1857
Epoch[ 10 / 50], Loss : 2.1947
Epoch[ 11 / 50], Loss : 2.1855
Epoch[ 12 / 50], Loss : 2.1860
Epoch[ 13 / 50], Loss : 2.1999
Epoch[ 14 / 50], Loss : 2.2082
Epoch[ 15 / 50], Loss : 2.1808
Epoch[ 16 / 50], Loss : 2.1986
Epoch[ 17 / 50], Loss : 2.1863
Epoch[ 18 / 50], Loss : 2.1940
Epoch[ 19 / 50], Loss : 2.1886
Epoch[ 20 / 50], Loss : 2.1779
Epoch[ 21 / 50], Loss : 2.1860
Epoch[ 22 / 50], Loss : 2.1854
Epoch[ 23 / 50], Loss : 2.1957
Epoch[ 24 / 50], Loss : 2.1854
Epoch[ 25 / 50], Loss : 2.1804
Epoch[ 26 / 50], Loss : 2.1811
Epoch[ 27 / 50], Loss : 2.1945
Epoch[ 28 / 50], Loss : 2.1961
Epoch[ 29 / 50], Loss : 2.1843
Epoch[ 30 / 50], Loss : 2.1858
Epoch[ 31 / 50], Loss : 2.1782
Epoch[ 32 / 50], Loss : 2.1942
Epoch[ 3

----
#### Model 1
: 입력층 -> 은닉층(활성화함수) -> 출력층

In [17]:
# 훈련 평가
train_loss, train_accuarcy = evaluate(model1, train_loader, criterion1, device)
print(f"Loss : {train_loss}, Accuracy : {train_accuarcy}")

Loss : 2.18861572154363, Accuracy : 0.8846666666666667


In [19]:
#검증 평가
val_loss, val_accuarcy = evaluate(model1, val_loader, criterion1, device)
print(f"Loss : {val_loss}, Accuracy : {val_accuarcy}")

Loss : 2.191457782745361, Accuracy : 0.8653333333333333


In [20]:
# 일반화 평가
test_dataset = TensorDataset(test_scaled, test_target)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


# 평가하기
test_loss, test_accuarcy = evaluate(model1, test_loader, criterion1, device)
print(f"Loss : {test_loss}, Accuracy : {test_accuarcy}")

Loss : 2.192584780458444, Accuracy : 0.8581


In [21]:
# Pandas로 정리하여 보기 위해 List로 정리하기
model1Result = [train_accuarcy, val_accuarcy, test_accuarcy]

In [ ]:
# 모델 인스턴스 
model2 = FashionMNISTModel1().to(device)
# 손실함수와 옵티마이져
criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(model2.parameters())

In [23]:
# Model 2 : 입력층 -> 은닉층(활성화함수)-> Dropout층 -> 출력층
class FashionMNISTModel2(nn.Module):
    def __init__(self):
        super(FashionMNISTModel2,self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,100)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(100,10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self,x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x =self.dropout1(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x


In [29]:
# Model 3 : 입력층 -> 은닉층(활성화함수)-> Dropout층 -> 은닉층(활성화함수)-> 출력층

class FashionMNISTModel3(nn.Module):
    def __init__(self):
        super(FashionMNISTModel3,self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,100)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(100,50)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(50,10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self,x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x =self.dropout1(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.softmax(x)
        return x


In [36]:
# Model 3 : 입력층 -> 은닉층(활성화함수)-> Dropout층 -> 은닉층(활성화함수)-> Dropout층 -> 출력층

class FashionMNISTModel4(nn.Module):
    def __init__(self):
        super(FashionMNISTModel4,self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,100)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(100,50)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(50,10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self,x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x =self.dropout1(x)
        x = self.fc2(x)
        x = self.relu(x)
        x =self.dropout2(x)
        x = self.fc3(x)
        x = self.softmax(x)
        return x


In [42]:
# 모델 인스턴스 
model2 = FashionMNISTModel2().to(device)
# 손실함수와 옵티마이져
criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(model2.parameters())

In [43]:
# 훈련하기 - 2
num_epochs = 50

for epoch in range(num_epochs):
    train_loss = train(model2, train_loader, criterion2, optimizer2, device)
    print(f'Epoch[{epoch+1:>3} / {num_epochs}], Loss : {train_loss:.4f}')

Epoch[  1 / 50], Loss : 1.6089
Epoch[  2 / 50], Loss : 1.6418
Epoch[  3 / 50], Loss : 1.6069
Epoch[  4 / 50], Loss : 1.6985
Epoch[  5 / 50], Loss : 1.5534
Epoch[  6 / 50], Loss : 1.6336
Epoch[  7 / 50], Loss : 1.7191
Epoch[  8 / 50], Loss : 1.7239
Epoch[  9 / 50], Loss : 1.6061
Epoch[ 10 / 50], Loss : 1.5619
Epoch[ 11 / 50], Loss : 1.5519
Epoch[ 12 / 50], Loss : 1.6677
Epoch[ 13 / 50], Loss : 1.5625
Epoch[ 14 / 50], Loss : 1.5956
Epoch[ 15 / 50], Loss : 1.6355
Epoch[ 16 / 50], Loss : 1.5956
Epoch[ 17 / 50], Loss : 1.5853
Epoch[ 18 / 50], Loss : 1.6465
Epoch[ 19 / 50], Loss : 1.5715
Epoch[ 20 / 50], Loss : 1.6015
Epoch[ 21 / 50], Loss : 1.5853
Epoch[ 22 / 50], Loss : 1.6063
Epoch[ 23 / 50], Loss : 1.6382
Epoch[ 24 / 50], Loss : 1.5670
Epoch[ 25 / 50], Loss : 1.6491
Epoch[ 26 / 50], Loss : 1.5247
Epoch[ 27 / 50], Loss : 1.6317
Epoch[ 28 / 50], Loss : 1.6432
Epoch[ 29 / 50], Loss : 1.5482
Epoch[ 30 / 50], Loss : 1.5561
Epoch[ 31 / 50], Loss : 1.6653
Epoch[ 32 / 50], Loss : 1.5901
Epoch[ 3

In [44]:
# 훈련 평가
train_loss, train_accuarcy = evaluate(model2, train_loader, criterion2, device)
print(f"Loss : {train_loss}, Accuracy : {train_accuarcy}")

#검증 평가
val_loss, val_accuarcy = evaluate(model2, val_loader, criterion2, device)
print(f"Loss : {val_loss}, Accuracy : {val_accuarcy}")

# 일반화 평가
test_dataset = TensorDataset(test_scaled, test_target)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# 평가하기
test_loss, test_accuarcy = evaluate(model2, test_loader, criterion2, device)
print(f"Loss : {test_loss}, Accuracy : {test_accuarcy}")


# Pandas로 정리하여 보기 위해 List로 정리하기
model3Result = [train_accuarcy, val_accuarcy, test_accuarcy]
model3Result

Loss : 1.5754469311237336, Accuracy : 0.8855625
Loss : 1.590177664756775, Accuracy : 0.87075
Loss : 1.597259278876332, Accuracy : 0.864


[0.8855625, 0.87075, 0.864]

In [26]:
# # Pandas로 정리하여 보기 위해 List로 정리하기
# model1Result = [train_accuarcy, val_accuarcy, test_accuarcy]
# model1Result


[0.8846666666666667, 0.8653333333333333, 0.8581]

----
#### Model 2
: 입력층 -> 은닉층(활성화함수) -> 출력층

In [28]:
# 훈련 평가
train_loss, train_accuarcy = evaluate(model2, train_loader, criterion2, device)
print(f"Loss : {train_loss}, Accuracy : {train_accuarcy}")

#검증 평가
val_loss, val_accuarcy = evaluate(model2, val_loader, criterion2, device)
print(f"Loss : {val_loss}, Accuracy : {val_accuarcy}")

# 일반화 평가
test_dataset = TensorDataset(test_scaled, test_target)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# 평가하기
test_loss, test_accuarcy = evaluate(model2, test_loader, criterion2, device)
print(f"Loss : {test_loss}, Accuracy : {test_accuarcy}")


# Pandas로 정리하여 보기 위해 List로 정리하기
model2Result = [train_accuarcy, val_accuarcy, test_accuarcy]
model2Result

Loss : 2.187154112815857, Accuracy : 0.8946458333333334
Loss : 2.1900180740356445, Accuracy : 0.8751666666666666
Loss : 2.191166491554187, Accuracy : 0.867


[0.8946458333333334, 0.8751666666666666, 0.867]

In [ ]:
# 모델 인스턴스 
model3 = FashionMNISTModel3().to(device)
# 손실함수와 옵티마이져
criterion3 = nn.CrossEntropyLoss()
optimizer3 = optim.Adam(model3.parameters())

In [46]:
# 훈련하기 - 3
num_epochs = 50

for epoch in range(num_epochs):
    train_loss = train(model3, train_loader, criterion3, optimizer3, device)
    print(f'Epoch[{epoch+1:>3} / {num_epochs}], Loss : {train_loss:.4f}')

Epoch[  1 / 50], Loss : 1.8137
Epoch[  2 / 50], Loss : 1.6721
Epoch[  3 / 50], Loss : 1.5416
Epoch[  4 / 50], Loss : 1.6475
Epoch[  5 / 50], Loss : 1.6399
Epoch[  6 / 50], Loss : 1.6803
Epoch[  7 / 50], Loss : 1.7046
Epoch[  8 / 50], Loss : 1.6055
Epoch[  9 / 50], Loss : 1.5480
Epoch[ 10 / 50], Loss : 1.6259
Epoch[ 11 / 50], Loss : 1.5530
Epoch[ 12 / 50], Loss : 1.5917
Epoch[ 13 / 50], Loss : 1.6449
Epoch[ 14 / 50], Loss : 1.6342
Epoch[ 15 / 50], Loss : 1.5897
Epoch[ 16 / 50], Loss : 1.7255
Epoch[ 17 / 50], Loss : 1.6073
Epoch[ 18 / 50], Loss : 1.6174
Epoch[ 19 / 50], Loss : 1.6159
Epoch[ 20 / 50], Loss : 1.7175
Epoch[ 21 / 50], Loss : 1.6796
Epoch[ 22 / 50], Loss : 1.6180
Epoch[ 23 / 50], Loss : 1.6476
Epoch[ 24 / 50], Loss : 1.5889
Epoch[ 25 / 50], Loss : 1.6831
Epoch[ 26 / 50], Loss : 1.5237
Epoch[ 27 / 50], Loss : 1.7449
Epoch[ 28 / 50], Loss : 1.6253
Epoch[ 29 / 50], Loss : 1.6913
Epoch[ 30 / 50], Loss : 1.6134
Epoch[ 31 / 50], Loss : 1.6807
Epoch[ 32 / 50], Loss : 1.5549
Epoch[ 3

In [47]:
# 훈련 평가
train_loss, train_accuarcy = evaluate(model3, train_loader, criterion3, device)
print(f"Loss : {train_loss}, Accuracy : {train_accuarcy}")

#검증 평가
val_loss, val_accuarcy = evaluate(model2, val_loader, criterion3, device)
print(f"Loss : {val_loss}, Accuracy : {val_accuarcy}")

# 일반화 평가
test_dataset = TensorDataset(test_scaled, test_target)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# 평가하기
test_loss, test_accuarcy = evaluate(model3, test_loader, criterion3, device)
print(f"Loss : {test_loss}, Accuracy : {test_accuarcy}")


# Pandas로 정리하여 보기 위해 List로 정리하기
model3Result = [train_accuarcy, val_accuarcy, test_accuarcy]
model3Result

Loss : 1.5904154465198517, Accuracy : 0.8703958333333334
Loss : 1.590177664756775, Accuracy : 0.87075
Loss : 1.6085239142274703, Accuracy : 0.8524


[0.8703958333333334, 0.87075, 0.8524]

In [39]:
# 모델 인스턴스 
model4 = FashionMNISTModel4().to(device)
# 손실함수와 옵티마이져
criterion4 = nn.CrossEntropyLoss()
optimizer4 = optim.Adam(model4.parameters())

In [40]:
# 훈련하기 - 4
num_epochs = 50

for epoch in range(num_epochs):
    train_loss = train(model4, train_loader, criterion4, optimizer4, device)
    print(f'Epoch[{epoch+1:>3} / {num_epochs}], Loss : {train_loss:.4f}')

Epoch[  1 / 50], Loss : 1.7051
Epoch[  2 / 50], Loss : 1.6877
Epoch[  3 / 50], Loss : 1.6468
Epoch[  4 / 50], Loss : 1.6205
Epoch[  5 / 50], Loss : 1.6816
Epoch[  6 / 50], Loss : 1.6568
Epoch[  7 / 50], Loss : 1.4978
Epoch[  8 / 50], Loss : 1.6497
Epoch[  9 / 50], Loss : 1.6817
Epoch[ 10 / 50], Loss : 1.5701
Epoch[ 11 / 50], Loss : 1.6149
Epoch[ 12 / 50], Loss : 1.5521
Epoch[ 13 / 50], Loss : 1.6852
Epoch[ 14 / 50], Loss : 1.7115
Epoch[ 15 / 50], Loss : 1.5865
Epoch[ 16 / 50], Loss : 1.7419
Epoch[ 17 / 50], Loss : 1.7198
Epoch[ 18 / 50], Loss : 1.6487
Epoch[ 19 / 50], Loss : 1.5976
Epoch[ 20 / 50], Loss : 1.6586
Epoch[ 21 / 50], Loss : 1.5010
Epoch[ 22 / 50], Loss : 1.7090
Epoch[ 23 / 50], Loss : 1.6591
Epoch[ 24 / 50], Loss : 1.6404
Epoch[ 25 / 50], Loss : 1.6064
Epoch[ 26 / 50], Loss : 1.6088
Epoch[ 27 / 50], Loss : 1.5982
Epoch[ 28 / 50], Loss : 1.5594
Epoch[ 29 / 50], Loss : 1.5237
Epoch[ 30 / 50], Loss : 1.7115
Epoch[ 31 / 50], Loss : 1.5549
Epoch[ 32 / 50], Loss : 1.5549
Epoch[ 3

In [41]:
# 훈련 평가
train_loss, train_accuarcy = evaluate(model4, train_loader, criterion3, device)
print(f"Loss : {train_loss}, Accuracy : {train_accuarcy}")

#검증 평가
val_loss, val_accuarcy = evaluate(model4, val_loader, criterion4, device)
print(f"Loss : {val_loss}, Accuracy : {val_accuarcy}")

# 일반화 평가
test_dataset = TensorDataset(test_scaled, test_target)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# 평가하기
test_loss, test_accuarcy = evaluate(model4, test_loader, criterion4, device)
print(f"Loss : {test_loss}, Accuracy : {test_accuarcy}")


# Pandas로 정리하여 보기 위해 List로 정리하기
model4Result = [train_accuarcy, val_accuarcy, test_accuarcy]
model4Result

Loss : 1.6026365032196046, Accuracy : 0.858375
Loss : 1.6135481112798056, Accuracy : 0.8474166666666667
Loss : 1.6190930692532572, Accuracy : 0.8418


[0.858375, 0.8474166666666667, 0.8418]